In [2]:
import requests
import pandas as pd
import time
from datetime import date
from sklearn.linear_model import LinearRegression
from scipy.stats import f_oneway
from datetime import timedelta
import os

In [3]:
tmdb_api_key = input("Enter TMDB API key: ")

In [11]:
DEFAULT_START_DATE = "2000-01-01"
END_DATE = "2025-12-31"


def get_movies(start_date=DEFAULT_START_DATE, end_date=END_DATE):
    tmdb_url = "https://api.themoviedb.org/3/discover/movie"
    all_movies = []
    page = 1

    while True:
        params = {
            "api_key": tmdb_api_key,
            "page": page,
            "primary_release_date.gte": start_date,
            "primary_release_date.lte": end_date,
            "sort_by": "primary_release_date.asc",
            "vote_count.gte": 1,
            "with_origin_country": "US",
        }

        try:
            response = requests.get(tmdb_url, params=params)
            response.raise_for_status()
            movie_data = response.json()
        except requests.exceptions.RequestException:
            break

        results = movie_data.get("results", [])
        if not results:
            break

        all_movies.extend(results)
        print(f"Fetched page {page}")

        total_pages = movie_data.get("total_pages", 1)
        if page >= total_pages or page >= 5:
            break

        page += 1

    return all_movies

In [12]:
def get_movie_info(movie_id):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    params = {"api_key": tmdb_api_key, "append_to_response": "credits"}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        info_get = response.json()
    except requests.exceptions.RequestException:
        return None

    if (info_get.get("budget") or 0) == 0 or (info_get.get("revenue") or 0) == 0:
        return None

    return {
        "tmdb_id": movie_id,
        "title": info_get.get("title"),
        "release_date": info_get.get("release_date"),
        "runtime": info_get.get("runtime"),
        "budget": info_get.get("budget") or 0,
        "revenue": info_get.get("revenue") or 0,
        "cast": [c["name"] for c in info_get.get("credits", {}).get("cast", [])[:10]],
        "crew": [c["name"] for c in info_get.get("credits", {}).get("crew", [])[:10]],
        "countries": [
            c["iso_3166_1"] for c in info_get.get("production_countries", [])
        ],
        "genres": [g["name"] for g in info_get.get("genres", [])],
        "production_companies": [
            c["name"] for c in info_get.get("production_companies", [])
        ],
    }

In [14]:
def get_resume_start_date(output_path):
    if not os.path.exists(output_path):
        return DEFAULT_START_DATE

    existing_df = pd.read_csv(output_path)

    if existing_df.empty or "release_date" not in existing_df.columns:
        return DEFAULT_START_DATE

    existing_df["release_date"] = pd.to_datetime(
        existing_df["release_date"], errors="coerce"
    )

    latest_date = existing_df["release_date"].max()

    if pd.isna(latest_date):
        return DEFAULT_START_DATE

    next_date = latest_date + timedelta(days=1)
    return next_date.strftime("%Y-%m-%d")


def main():
    output_path = "../data/tmdb_movies.csv"

    start_date = get_resume_start_date(output_path)
    print(f"Starting from: {start_date}")

    dataset = []

    for movie in get_movies(start_date=start_date, end_date=END_DATE):
        movie_id = movie["id"]
        details = get_movie_info(movie_id)
        time.sleep(0.0001)

        print(f"Processing movie {movie_id}")

        if details:
            dataset.append(details)

    if not dataset:
        print("No new movies found.")
        return pd.DataFrame()

    df = pd.DataFrame(dataset)

    if os.path.exists(output_path):
        df.to_csv(output_path, mode="a", header=False, index=False)
    else:
        df.to_csv(output_path, index=False)

    print(f"Added {len(df)} rows to {output_path}")
    return df


if __name__ == "__main__":
    df = main()

Starting from: 2004-10-14
Fetched page 1
Fetched page 2
Fetched page 3
Fetched page 4
Fetched page 5
Processing movie 695902
Processing movie 621994
Processing movie 396088
Processing movie 354628
Processing movie 67279
Processing movie 49132
Processing movie 48331
Processing movie 41569
Processing movie 504151
Processing movie 146546
Processing movie 34359
Processing movie 31165
Processing movie 29620
Processing movie 25313
Processing movie 16791
Processing movie 16358
Processing movie 12483
Processing movie 11099
Processing movie 4380
Processing movie 4237
Processing movie 2319
Processing movie 1398493
Processing movie 919646
Processing movie 429876
Processing movie 360703
Processing movie 246353
Processing movie 179263
Processing movie 152848
Processing movie 97213
Processing movie 88001
Processing movie 73452
Processing movie 32707
Processing movie 743919
Processing movie 606121
Processing movie 167177
Processing movie 27860
Processing movie 308124
Processing movie 135261
Processin

In [ ]:
# df = df.drop_duplicates(subset=["tmdb_id"])
# df = df[(df["budget"] > 0) & (df["revenue"] > 0)]
# df = df[df["countries"].apply(lambda x: isinstance(x, list) and "US" in x)]

KeyError: 'budget'

In [ ]:
df_cpi = pd.read_excel("historical-cpi-u-202603.xlsx", skiprows=3)

df_cpi = df_cpi.drop(columns=["Indent Level"])

df_cpi.columns = [
    "year",
    "jan",
    "feb",
    "mar",
    "apr",
    "may",
    "jun",
    "jul",
    "aug",
    "sep",
    "oct",
    "nov",
    "dec",
]

df_cpi["year"] = pd.to_numeric(df_cpi["year"], errors="coerce")
df_cpi = df_cpi.dropna(subset=["year"])

month_cols = df_cpi.columns[1:]
df_cpi[month_cols] = df_cpi[month_cols].apply(pd.to_numeric, errors="coerce")

df_cpi = df_cpi[(df_cpi["year"] >= 2000) & (df_cpi["year"] <= 2026)]

df_cpi["cpi"] = df_cpi[month_cols].mean(axis=1)

df_cpi = df_cpi[["year", "cpi"]]

df["release_year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year

df = df.merge(df_cpi, left_on="release_year", right_on="year", how="left")

df = df[df["cpi"].notna()]

base_cpi = df_cpi.loc[df_cpi["year"] == 2025, "cpi"].iloc[0]

df["budget_adj"] = df["budget"] * (base_cpi / df["cpi"])
df["revenue_adj"] = df["revenue"] * (base_cpi / df["cpi"])

df["roi_adj"] = (df["revenue_adj"] - df["budget_adj"]) / df["budget_adj"]

print(df[["budget", "budget_adj", "revenue", "revenue_adj"]].head())
print("Missing CPI values:", df["cpi"].isna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'historical-cpi-u-202603.xlsx'

In [ ]:
df = df[df["cpi"].notna()]
df = df[df["budget_adj"] > 0]
df = df[df["revenue_adj"] > 0]
df = df.dropna(subset=["roi_adj"])